### Write Motor IDs in terminal:
uv run lerobot-find-port

lerobot-setup-motors --teleop.type dk1_leader --teleop.port YOUR_PORT

## Calibration init

In [ ]:
from lerobot_robot_trlc_dk1.leader import DK1Leader, DK1LeaderConfig
import time
from IPython.display import clear_output
import ipywidgets as widgets

leader_config = DK1LeaderConfig(
    port="/dev/tty.usbmodem59700725851"
)

leader = DK1Leader(leader_config)
leader.connect()
leader.bus.write("Torque_Enable", "gripper", 0, normalize=False)


# <font color='red'>PLACE LEADER IN RESTING POSITION</font>

In [ ]:
leader.bus.sync_write("Torque_Enable", 0, normalize=False)
leader.bus.sync_write("Homing_Offset", 0, normalize=False)
joints_to_calibrate = ["joint_2", "joint_3", "joint_4"]
theoretical_zero_position = {"joint_2": 2048, "joint_3": 2048, "joint_4": 2048}

for joint in joints_to_calibrate:
    position = leader.bus.read("Present_Position", joint, normalize=False)
    offset = theoretical_zero_position[joint] - position
    print(f"{joint}: Position: {position}, Offset: {offset}")
    leader.bus.write("Homing_Offset", joint, offset, normalize=False)

leader.bus.sync_read("Homing_Offset", normalize=False)

print(leader.bus.sync_read(normalize=False, data_name="Present_Position"))

print("Joint 2-4 should be at 2048 +- 2")

## Test joint angles

In [ ]:
from IPython.display import display, update_display, HTML
import time
display_handle = display(HTML("<pre>Action: Initializing...</pre>"), display_id="leader_watch")

# try:
start_time = time.time()
while (time.time()-start_time) < 15:
    data = leader.get_action()
    lines = []
    for k, v in data.items():
        lines.append(f"{k}:\t{v:.2f}")
    full_text = "<pre>" + "\n".join(lines) + "</pre>"
    update_display(HTML(full_text), display_id="leader_watch")
    time.sleep(.05)
# except KeyboardInterrupt:
#     update_display(HTML("<pre>Watch stopped.</pre>"), display_id="leader_watch")